# M3L3 E22 - E08 con Langfuse: ver trazas en un router

Objetivo: tomar el router condicional de E08 y observar en Langfuse la llamada que clasifica la intención.

Este ejercicio muestra algo más útil que E21: no solo vemos una respuesta, vemos una decisión de routing tomada por el LLM.

## Paso 1 - Conectar OpenAI y Langfuse

Este bloque es lo mínimo para que una llamada al LLM aparezca en Langfuse.

Qué vamos a hacer:
- Instalar LangGraph, LangChain OpenAI y Langfuse.
- Cargar la API key de OpenAI con `getpass`.
- Cargar las keys de Langfuse con `getpass`.
- Configurar `LANGFUSE_BASE_URL` para US Cloud.
- Crear `CallbackHandler`, que es el puente entre LangChain y Langfuse.

No guardamos ninguna key dentro del notebook. Cada estudiante la pega en runtime.

In [ ]:
!pip install langgraph langchain langchain-openai langfuse -q

import os
from getpass import getpass
from typing import TypedDict

from langchain_openai import ChatOpenAI
from langfuse import Langfuse
from langfuse.langchain import CallbackHandler
from langgraph.graph import StateGraph, START, END

os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ").strip()
os.environ["LANGFUSE_PUBLIC_KEY"] = getpass("Langfuse Public Key: ").strip()
os.environ["LANGFUSE_SECRET_KEY"] = getpass("Langfuse Secret Key: ").strip()

# US Cloud. Si tu proyecto está en EU, usar: https://cloud.langfuse.com
langfuse_base_url = input("Langfuse Base URL [https://us.cloud.langfuse.com]: ").strip() or "https://us.cloud.langfuse.com"
langfuse_base_url = langfuse_base_url.rstrip("/")
os.environ["LANGFUSE_BASE_URL"] = langfuse_base_url
os.environ["LANGFUSE_HOST"] = langfuse_base_url  # compatibilidad con integraciones anteriores

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
langfuse = Langfuse()

if not langfuse.auth_check():
    raise RuntimeError("Langfuse no autenticó. Revisar keys y Base URL.")

langfuse_handler = CallbackHandler()


def invoke_llm(prompt: str):
    # Todas las llamadas que pasen por acá quedan trazadas en Langfuse.
    return llm.invoke(prompt, config={"callbacks": [langfuse_handler]})

print("OpenAI + Langfuse listos")

## Paso 2 - Knowledge base mínima

Cada categoría tiene respuestas fijas. El LLM no responde el ticket final: solo clasifica la intención.

Esto permite explicar Langfuse con claridad: la traza importante es la clasificación.

In [ ]:
knowledge_base = {
    "hr": ["Vacaciones: 15 días hábiles por año.", "Beneficios: prepaga y home office 2 días por semana."],
    "tech": ["VPN: reiniciar cliente y validar MFA.", "SaaS aprobados: Slack, GitHub, Jira, Figma."],
    "billing": ["Facturas: se procesan los primeros 5 días hábiles.", "Reembolsos: requieren comprobante."],
}


def retrieve(intent: str, query: str) -> list[str]:
    return knowledge_base.get(intent, [])

print("Knowledge base cargada")

## Paso 3 - State del router

El estado guarda:
- `query`: pregunta original.
- `intent`: categoría elegida por el LLM.
- `response`: respuesta final del nodo especializado.

La parte observable en Langfuse es la llamada dentro de `classify(...)`.

In [ ]:
class RouterState(TypedDict):
    query: str
    intent: str
    response: str

print("State definido")

## Paso 4 - Nodo clasificador y nodos destino

`classify` llama al LLM para decidir entre `hr`, `tech`, `billing` o `unknown`.

Los demás nodos son determinísticos: toman la categoría y devuelven una respuesta desde la knowledge base.

In [ ]:
def classify(state: RouterState) -> dict:
    prompt = (
        "Clasificá esta consulta de soporte corporativo en exactamente una categoría.\n"
        "Categorías válidas: hr, tech, billing, unknown\n"
        "Respondé SOLO con la categoría, sin explicación ni puntuación.\n\n"
        f"Consulta: {state['query']}"
    )
    response = invoke_llm(prompt)
    intent = response.content.strip().lower()
    if intent not in ("hr", "tech", "billing"):
        intent = "unknown"
    return {"intent": intent}


def hr_node(state: RouterState) -> dict:
    return {"response": "HRAgent: " + " | ".join(retrieve("hr", state["query"]))}


def tech_node(state: RouterState) -> dict:
    return {"response": "TechAgent: " + " | ".join(retrieve("tech", state["query"]))}


def billing_node(state: RouterState) -> dict:
    return {"response": "BillingAgent: " + " | ".join(retrieve("billing", state["query"]))}


def unknown_node(state: RouterState) -> dict:
    return {"response": "No puedo responder esa consulta."}


def router(state: RouterState) -> str:
    return {"hr": "hr_node", "tech": "tech_node", "billing": "billing_node"}.get(
        state["intent"], "unknown_node"
    )

print("Nodos definidos")


## Paso 5 - Compilar el grafo condicional

El flujo es:

`START -> classify -> rama condicional -> END`

La decisión condicional depende de `intent`, que fue producido por el LLM.

In [ ]:
graph = StateGraph(RouterState)
graph.add_node("classify", classify)
graph.add_node("hr_node", hr_node)
graph.add_node("tech_node", tech_node)
graph.add_node("billing_node", billing_node)
graph.add_node("unknown_node", unknown_node)

graph.add_edge(START, "classify")
graph.add_conditional_edges(
    "classify",
    router,
    {
        "hr_node": "hr_node",
        "tech_node": "tech_node",
        "billing_node": "billing_node",
        "unknown_node": "unknown_node",
    },
)

for node in ["hr_node", "tech_node", "billing_node", "unknown_node"]:
    graph.add_edge(node, END)

app = graph.compile()
print("Grafo E22 compilado")

## Paso 6 - Ejecutar varias consultas y mirar Langfuse

Cada consulta genera una llamada al LLM dentro de `classify(...)`. En Langfuse deberías ver varias generations, una por clasificación.

Qué contar en clase:
- Esta traza muestra por qué el sistema fue a HR, Tech o Billing.
- Si el router clasifica mal, Langfuse nos deja ver el input exacto que causó el error.
- Esto sirve para debuggear prompts y decisiones de routing.

In [ ]:
queries = [
    "Cuántas vacaciones tengo",
    "No me funciona la VPN",
    "Cuándo se procesa mi factura",
    "Qué hay para almorzar",
]

for q in queries:
    r = app.invoke({"query": q, "intent": "", "response": ""})
    print(f"[{r['intent']:8s}] {q} -> {r['response'][:80]}")

langfuse.flush()
print("Listo: abrí Langfuse -> Tracing y compará las trazas del router")

## Cómo explicarlo

Frase corta para clase:

> En un router con LLM, Langfuse nos permite auditar decisiones. No miramos solo la respuesta final; miramos el paso donde el modelo decidió a qué agente mandar la consulta.

Lo importante de este ejercicio:
- `classify` es el nodo observable.
- LangGraph usa esa decisión para elegir una rama.
- Langfuse permite revisar inputs, outputs, tokens y latencia de cada clasificación.